# 🎓 SAGE — AI Academic Companion  
### Final Year Project Demonstration Notebook

**Author:** *(your name)* &nbsp;·&nbsp; **Supervisor:** *(supervisor name)* &nbsp;·&nbsp; **Module:** Final Year Project

---

## Abstract

**SAGE** (*Study Assistant with Generative Embeddings*) is a **fully local, privacy-preserving AI academic companion** designed to run on a typical student laptop with no cloud dependency. It combines three tightly integrated study features around a single Retrieval-Augmented Generation (RAG) core:

| # | Feature | Pipeline |
|---|---------|----------|
| 1 | 🎙️ **Voice Note Summarizer** | Mic → Moonshine STT → RAG → structured study notes |
| 2 | 🃏 **Flashcard Generator** | Transcript → RAG → parsed Q/A cards |
| 3 | 📄 **Document Summarizer** | `.docx` / `.pptx` → LangChain loader → RAG → structured summary |

### Technology stack

| Layer | Component | Rationale |
|---|---|---|
| LLM (default) | **TinyLlama 1.1B** via **Ollama** | ~640 MB, fits in ≤2 GB RAM, runs on any student laptop |
| LLM (optional) | **Phi-3.5-mini** (3.8 B) or **Mistral 7B Instruct** | Higher answer quality when ≥4 GB free RAM is available — auto-selected via `OLLAMA_FALLBACK_MODELS` |
| Embeddings | **all-MiniLM-L6-v2** (Sentence-Transformers) | 22 MB, CPU-friendly, 384-dim |
| Vector DB | **ChromaDB** (persistent, local) | Zero-config, file-backed |
| Orchestration | **LangChain** (`RetrievalQA`) | Standard, auditable RAG chain |
| STT | **Moonshine ONNX** | Faster than Whisper on CPU |
| GUI | **CustomTkinter** | Native look, no browser needed |

The LLM is pluggable: `config.OLLAMA_MODEL` (or the `OLLAMA_MODEL` environment variable) selects the active model, and `config.OLLAMA_FALLBACK_MODELS` provides automatic upgrade paths when more RAM is available. Every other component in the pipeline is **model-agnostic**.

### What this notebook demonstrates

This notebook bypasses the CustomTkinter GUI and exercises every backend component **head-on**, so an examiner can verify each subsystem in isolation and end-to-end:

1. System health & configuration  
2. Embedding model  
3. Vector store  
4. Core RAG pipeline  
5. Voice-note summarisation (text path)  
6. Flashcard generation & robust parsing  
7. Document summarisation (`.docx`)  
8. Quantitative evaluation (timings, throughput)  
9. Artefact inventory on disk  

> ⚠️ **Prerequisite:** `ollama serve` must be running and the default model pulled with `ollama pull tinyllama` (≈640 MB). On first STT use, Moonshine downloads the selected ONNX model files and may take extra time. For higher-quality output on machines with more RAM, you may additionally pull `ollama pull phi3.5` (≈2.2 GB) and either set `OLLAMA_MODEL=phi3.5` or rely on the configured fallback chain. The notebook performs an automatic health check in §1 and degrades gracefully if Ollama is unreachable.



## Section 0 — Environment setup

We add the SAGE project root to `sys.path` so the notebook can import the same modules the GUI uses (`config`, `rag.*`, `features.*`, `utils.*`). This guarantees **what is demonstrated here is exactly what runs in production**, not a parallel re-implementation.


In [ ]:
import os, sys, time, warnings
from pathlib import Path

warnings.filterwarnings("ignore")  # silence LangChain deprecation chatter for cleaner demo output

# The notebook lives at the project root (next to config.py, features/, rag/, utils/).
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root  : {PROJECT_ROOT}")
print(f"Python        : {sys.version.split()[0]}")
print(f"Working dir   : {os.getcwd()}")

# Sanity-check that the expected packages exist on disk
for pkg in ("config.py", "rag", "features", "utils"):
    exists = (PROJECT_ROOT / pkg).exists()
    print(f"  [{'OK' if exists else '!!'}] {pkg}")


## Section 1 — Health check (Ollama + Phi-3.5)

SAGE refuses to start the GUI if Ollama is unreachable; the same guard is invoked here. The function `utils.health_check.check_ollama()`:

1. pings `http://localhost:11434`,
2. queries `/api/tags` and looks for the configured model,
3. returns `(ok: bool, message: str)` with a user-friendly fix-it message on failure.

If the check fails, the notebook **continues** but every LLM-dependent cell will print a `SKIPPED` banner instead of crashing — so non-LLM sections (embeddings, vector store, parser tests, artefact listing) still demonstrate cleanly.


In [ ]:
from utils.health_check import check_ollama
from config import OLLAMA_MODEL

t0 = time.perf_counter()
OLLAMA_OK, msg = check_ollama(OLLAMA_MODEL)
dt = time.perf_counter() - t0

banner = "=" * 60
print(banner)
if OLLAMA_OK:
    print(f"  ✅ OLLAMA HEALTHY  —  model '{OLLAMA_MODEL}' available")
    print(f"  Check completed in {dt:.2f}s")
else:
    print(f"  ❌ OLLAMA UNAVAILABLE")
    print()
    print(msg)
    print()
    print("  LLM-dependent sections (§3, §4, §5, §7) will be SKIPPED.")
    print("  Non-LLM sections (§2 embeddings, §6 parser, §8 artefacts) still run.")
print(banner)


### Active configuration

All tunables live in `config.py` (single source of truth). Below we print the values that drive every downstream cell — this is what an evaluator would change to reproduce the experiments with a different model or chunk size.


In [ ]:
import config
import pandas as pd
from IPython.display import display

cfg_rows = [
    ("LLM model",           config.OLLAMA_MODEL),
    ("LLM temperature",     config.OLLAMA_TEMP),
    ("LLM context window",  config.OLLAMA_CTX),
    ("Embedding model",     config.EMBEDDING_MODEL),
    ("Chunk size (chars)",  config.CHUNK_SIZE),
    ("Chunk overlap",       config.CHUNK_OVERLAP),
    ("Retriever k (text)",  config.RETRIEVER_K),
    ("Retriever k (docs)",  config.RETRIEVER_K_DOC),
    ("Moonshine model",     config.MOONSHINE_MODEL),
    ("Audio sample rate",   f"{config.SAMPLE_RATE} Hz"),
    ("ChromaDB dir",        os.path.relpath(config.CHROMA_DIR, PROJECT_ROOT)),
    ("Storage dir",         os.path.relpath(config.STORAGE_DIR, PROJECT_ROOT)),
]
cfg_df = pd.DataFrame(cfg_rows, columns=["Setting", "Value"])
display(cfg_df.style.hide(axis="index").set_caption("SAGE — Active configuration"))


## Section 2 — Embedding model

`rag/embeddings.py` exposes a **singleton** wrapper around `all-MiniLM-L6-v2`. The first call downloads / loads the model into RAM; every subsequent call returns the same instance — critical for keeping the GUI responsive.

We verify:
- the singleton actually caches (second call should be ~free),
- the produced vectors have the expected **384** dimensions,
- the vectors are L2-normalised (`config` sets `normalize_embeddings=True`), so cosine similarity is equivalent to a dot product.


In [ ]:
import numpy as np
from rag.embeddings import get_embeddings

# First call → cold load
t0 = time.perf_counter()
emb = get_embeddings()
cold = time.perf_counter() - t0

# Second call → should hit the singleton cache
t0 = time.perf_counter()
emb2 = get_embeddings()
warm = time.perf_counter() - t0

assert emb is emb2, "Singleton broken — second call returned a different instance"

sample = "Photosynthesis converts light energy into chemical energy stored in glucose."
t0 = time.perf_counter()
vec = np.array(emb.embed_query(sample))
encode_dt = time.perf_counter() - t0

print(f"Cold load        : {cold:6.2f}s")
print(f"Warm load        : {warm*1000:6.2f}ms  (singleton works)")
print(f"Encode 1 sentence: {encode_dt*1000:6.2f}ms")
print(f"Vector dim       : {vec.shape[0]}")
print(f"L2 norm          : {np.linalg.norm(vec):.4f}   (≈1.0 → normalised)")
print(f"First 8 dims     : {np.round(vec[:8], 4).tolist()}")
